In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
bronze_website=spark.read.format("delta").load("s3://retail-lakehouse-ashu/bronze/website/")

In [0]:
bronze_website.printSchema()

In [0]:
bronze_website.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
fact_orders=bronze_website.select(col("order_id"),col("customer_id"),col("order_status"),col("payment_amount"),col("order_timestamp").cast("timestamp"),col("coupon"),col("shipping_address.city").alias("shipping_city"),col("shipping_address.pincode").cast("long").alias("shipping_pincode"),col("shipping_address.state").alias("shipping_state"),col("ingestion_timestamp"),col("batch_id"),col("source_system"),col("source_file"))

In [0]:
fact_orders.printSchema()

In [0]:
bronze_website_explode=bronze_website.withColumn("items", explode_outer("items"))
bronze_website_explode.display()

In [0]:
bronze_website_explode.groupBy(
    "order_id",
    "items.product_id"
).count().filter(
    col("count") > 1
).show()

In [0]:
fact_order_items=bronze_website_explode.select(col("order_id"),col("customer_id"),col("items.product_id").alias("product_id"),col("items.product_name").alias("product_name"),col("items.quantity").alias("quantity"),col("items.price").alias("price"))
fact_order_items.display()

In [0]:
input_orders=bronze_website.count()
output_orders=fact_orders.count()
input_order_items=bronze_website_explode.count()
output_order_items=fact_order_items.count()
print("Input orders", input_orders)
print("Output orders", output_orders)
print("Input order items", input_order_items)
print("Output order items", output_order_items)

In [0]:
fact_orders.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/website/orders/")
fact_order_items.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/website/order_items/")